1.1 Ticker selection and data acquisition

In [ ]:
%pip install -q yfinance lxml

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timezone
from google.colab import drive
from pathlib import Path

In [ ]:
# Connect Google Drive
drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/CRM_CWK1_Data")
folder.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Get S&P 500 tickers from Wikipedia
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
sp500 = pd.read_html(
    url, storage_options={"User-Agent": "Mozilla/5.0"}
)[0]

retrieval_date = datetime.now(timezone.utc).date().isoformat()
sp500["Retrieval_Date"] = retrieval_date
sp500["Yahoo_Ticker"] = sp500["Symbol"].str.replace(
    ".", "-", regex=False
)
tickers = sp500["Yahoo_Ticker"].tolist()

In [ ]:
# Save ticker list
sp500.to_csv(folder / "sp500_constituents.csv", index=False)

In [ ]:
# Download daily stock data and S&P 500 benchmark
data = yf.download(
    tickers + ["^GSPC"],
    start="2021-01-01",
    end="2025-01-01",
    interval="1d",
    auto_adjust=False,
    actions=True,
    progress=False)

ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['FDXF', 'SNDK', 'HONA', 'VYLR', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


In [ ]:
# Check essential fields
assert not data.empty, "No data downloaded."

required = {"Close", "Adj Close", "Volume"}
fields = set(data.columns.get_level_values(0))
assert required.issubset(fields), "Required data fields are missing."

In [ ]:
# Save original data
data.to_csv(
    folder / "sp500_daily_2021_2024.csv.gz",
    compression="gzip")

In [ ]:

# Identify failed or empty downloads
available = (
    data["Close"].notna().any()
    & data["Adj Close"].notna().any()
    & data["Volume"].notna().any())

failed = sorted(set(tickers) - set(available[available].index))
pd.DataFrame({"Ticker": failed}).to_csv(
    folder / "download_failures.csv", index=False)

In [ ]:
# Verify results
print("Retrieval date:", retrieval_date)
print("Requested stocks:", len(tickers))
print("Stocks with data:", len(tickers) - len(failed))
print("First date:", data.index.min())
print("Last date:", data.index.max())
print("Data fields:", sorted(fields))
print("S&P 500 benchmark available:", bool(available.get("^GSPC", False)))
print("Failed downloads:", failed)
print("yfinance version:", yf.__version__)

Retrieval date: 2026-11-11
Requested stocks: 503
Stocks with data: 498
First date: 2021-01-04 00:00:00
Last date: 2024-12-31 00:00:00
Data fields: ['Adj Close', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock Splits', 'Volume']
S&P 500 benchmark available: True
Failed downloads: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
yfinance version: 0.2.66


In [ ]:
display(data["Close"].iloc[:5, :5])

Ticker,A,AAPL,ABBV,ABNB,ABT
Date,,,,,
2021-01-04,118.639999,129.410004,105.410004,139.149994,109.110001
2021-01-05,119.610001,131.009995,106.500000,148.300003,110.459999
2021-01-06,122.889999,126.599998,105.580002,142.770004,110.230003
2021-01-07,126.160004,130.919998,106.709999,151.270004,111.300003
2021-01-08,127.059998,132.050003,107.269997,149.770004,111.610001


1.2 Data Validation and Alignment

In [ ]:
# Check duplicate dates
duplicates = data.index.duplicated().sum()
print("Duplicate dates:", duplicates)


if duplicates:
    raise ValueError("Duplicate dates need investigation.")

Duplicate dates: 0


In [ ]:
# Match shares to benchmark trading dates
market_days = data["Adj Close"]["^GSPC"].dropna().index

close = data["Close"].reindex(index=market_days, columns=tickers)
adj = data["Adj Close"].reindex(index=market_days, columns=tickers)
volume = data["Volume"].reindex(index=market_days, columns=tickers)

In [ ]:
# Check data quality without filling missing values
valid = close.gt(0) & adj.gt(0) & volume.gt(0)

checks = pd.DataFrame(index=tickers)
checks["Valid_days"] = valid.sum()
checks["Missing_days"] = (
    close.isna() | adj.isna() | volume.isna()
).sum()
checks["Bad_prices"] = (close.le(0) | adj.le(0)).sum()
checks["Bad_volume"] = volume.le(0).sum()

In [ ]:
# Check unequal histories
checks["First_date"] = adj.where(valid).apply(pd.Series.first_valid_index)
checks["Last_date"] = adj.where(valid).apply(pd.Series.last_valid_index)

In [ ]:
# Check recorded corporate actions
for field in ["Dividends", "Stock Splits"]:
    if field in data.columns.get_level_values(0):
        checks[field.replace(" ", "_")] = (
            data[field]
            .reindex(index=market_days, columns=tickers)
            .fillna(0).ne(0).sum()
        )

print("Benchmark trading days:", len(market_days))
print("Shares with missing data:", (checks["Missing_days"] > 0).sum())
print("Shares with bad prices:", (checks["Bad_prices"] > 0).sum())
print("Shares with bad volume:", (checks["Bad_volume"] > 0).sum())

Benchmark trading days: 1005
Shares with missing data: 15
Shares with bad prices: 0
Shares with bad volume: 2


In [ ]:
display(checks.sort_values("Missing_days", ascending=False).head(15))

,Valid_days,Missing_days,Bad_prices,Bad_volume,First_date,Last_date,Dividends,Stock_Splits
HONA,0,1005,0,0,NaT,NaT,0,0
FDXF,0,1005,0,0,NaT,NaT,0,0
Q,0,1005,0,0,NaT,NaT,0,0
SNDK,0,1005,0,0,NaT,NaT,0,0
VYLR,0,1005,0,0,NaT,NaT,0,0
GEV,193,812,0,0,2024-03-27,2024-12-31,1,0
SOLV,194,811,0,0,2024-03-26,2024-12-31,0,0
RDDT,197,808,0,0,2024-03-21,2024-12-31,0,0
VLTO,313,692,0,0,2023-10-04,2024-12-31,5,0
GEHC,513,492,0,0,2022-12-15,2024-12-31,7,0


In [ ]:

# Weekly alignment using adjusted closing prices

weekly_prices = (
    adj.where(adj.gt(0) & close.gt(0))
    .resample("W-FRI").last()
    .loc["2021-01-08":"2024-12-27"])

market_weekly_prices = (
    data["Adj Close"]["^GSPC"]
    .reindex(market_days)
    .resample("W-FRI").last()
    .reindex(weekly_prices.index))

weekly_returns = weekly_prices.pct_change(fill_method=None)
market_weekly_returns = market_weekly_prices.pct_change(fill_method=None)


In [ ]:
# Count weeks with matching valid returns
checks["Aligned_weeks"] = (
    weekly_returns.notna()
    .mul(market_weekly_returns.notna(), axis=0)
    .sum())

In [ ]:
# Review history coverage
checks["Coverage_pct"] = (valid.mean() * 100).round(1)
checks["Last80_valid"] = valid.tail(80).all()

print("Weekly dates aligned:",
      weekly_returns.index.equals(market_weekly_returns.index))

print("Weekly date range:",
      weekly_returns.index.min(),
      "to", weekly_returns.index.max())

Weekly dates aligned: True
Weekly date range: 2021-01-08 00:00:00 to 2024-12-27 00:00:00


In [ ]:
display(checks[["Coverage_pct", "Aligned_weeks", "Last80_valid"]].describe())

,Coverage_pct,Aligned_weeks
count,503.000000,503.000000
mean,98.088270,203.178926
std,12.432668,25.508295
min,0.000000,0.000000
25%,100.000000,207.000000
50%,100.000000,207.000000
75%,100.000000,207.000000
max,100.000000,207.000000


In [ ]:
# Save validation evidence
checks.to_csv(folder / "validation_results.csv")

1.3 Weekly return and weekly beta

1.3.1 Mean weekly return & 1.3.2 Weekly beta

In [ ]:
results = []
for ticker in tickers:
    # Align stock and benchmark weekly returns
    paired = pd.concat([
        weekly_returns[ticker].rename("Stock"),
        market_weekly_returns.rename("Market")
    ], axis=1).replace([np.inf, -np.inf], np.nan).dropna()

    # Mean weekly return
    mean_return = paired["Stock"].mean()

    # Weekly beta
    beta = np.nan
    market_variance = paired["Market"].var()

    if len(paired) >= 52 and market_variance > 0:
        beta = paired["Stock"].cov(paired["Market"]) / market_variance

    results.append({
        "Ticker": ticker,
        "Mean_Weekly_Return": mean_return,
        "Weekly_Beta": beta,
        "Aligned_Weeks": len(paired)
    })

features = pd.DataFrame(results).set_index("Ticker")

In [ ]:
# Display and save results
display(features.head(10))

print("Maximum aligned weeks:", features["Aligned_Weeks"].max())
print("Stocks with valid beta:", features["Weekly_Beta"].notna().sum())

features.to_csv(folder / "weekly_return_beta_features.csv")
weekly_returns.to_csv(folder / "weekly_returns.csv")
market_weekly_returns.to_csv(folder / "benchmark_weekly_returns.csv")


,Mean_Weekly_Return,Weekly_Beta,Aligned_Weeks
Ticker,,,
MMM,0.001142,0.869080,207
AOS,0.001991,1.042804,207
ABT,0.000971,0.583543,207
ABBV,0.003749,0.425366,207
ACN,0.002376,1.143301,207
ADBE,0.000990,1.553525,207
AMD,0.003624,1.877622,207
AES,-0.001420,1.143912,207
AFL,0.005004,0.707197,207


Maximum aligned weeks: 207
Stocks with valid beta: 495


In [ ]:

# Independently verify beta for AAPL

paired = pd.concat([
    weekly_returns["AAPL"],
    market_weekly_returns
], axis=1).dropna()

# Regression slope is another way to calculate beta
beta_check = np.polyfit(paired.iloc[:, 1], paired.iloc[:, 0], 1)[0]

print("Original beta:", features.loc["AAPL", "Weekly_Beta"])
print("Verified beta:", beta_check)
print("Aligned weeks:", len(paired))

assert np.isclose(
    features.loc["AAPL", "Weekly_Beta"],
    beta_check
)

Original beta: 1.1748354096209954
Verified beta: 1.1748354096209959
Aligned weeks: 207


1.4 Adapted RFM Calculations

In [ ]:
#Identify unusually high-volume days
observation_date = pd.Timestamp("2024-12-31")

In [ ]:
# Use valid trading volume
aapl_volume = volume["AAPL"].where(volume["AAPL"] > 0)

In [ ]:
# Mean of the previous 20 trading sessions
previous_20_avg = aapl_volume.shift(1).rolling(
    20, min_periods=20
).mean()

In [ ]:
# Flag unusually high-volume days
high_volume = aapl_volume > (2 * previous_20_avg)

In [ ]:
# Inspect the last five sessions
display(pd.DataFrame({
    "Volume": aapl_volume,
    "Previous_20_Avg": previous_20_avg,
    "High_Volume": high_volume
}).loc[:observation_date].tail(5))


,Volume,Previous_20_Avg,High_Volume
Date,,,
2024-12-24,23234700,50408480.0,False
2024-12-26,27237100,47062575.0,False
2024-12-27,42355300,46125120.0,False
2024-12-30,35557500,46567965.0,False
2024-12-31,39480700,46921770.0,False


1.4.1 Recency

In [ ]:
# Find all qualifying high-volume event dates
event_dates = high_volume.loc[:observation_date]
event_dates = event_dates.index[event_dates]

if len(event_dates) > 0:
    last_event = event_dates[-1]
    recency = (observation_date - last_event).days

    print("Last high-volume event:", last_event.date())
    print("Event volume:", aapl_volume.loc[last_event])
    print("Previous 20-session average:", previous_20_avg.loc[last_event])
    print("Recency (calendar days):", recency)

else:
    recency = np.nan
    print("No qualifying event found. Recency:", recency)

Last high-volume event: 2024-12-20
Event volume: 147495300
Previous 20-session average: 45004605.0
Recency (calendar days): 11


1.4.2 Frequency

In [ ]:
# Select the latest 60 trading sessions
last_60_events = high_volume.loc[:observation_date].tail(60)

# Count qualifying high-volume days
frequency = int(last_60_events.sum())

print("Window start:", last_60_events.index[0].date())
print("Window end:", last_60_events.index[-1].date())
print("Trading sessions:", len(last_60_events))
print("Frequency:", frequency)

# Display qualifying event dates
display(last_60_events[last_60_events])

Window start: 2024-10-07
Window end: 2024-12-31
Trading sessions: 60
Frequency: 2


,AAPL
Date,
2024-11-25,True
2024-12-20,True


1.4.3 Monetary

In [ ]:
# Select the latest 60 trading sessions
last_60 = pd.DataFrame({
    "Close": close["AAPL"],
    "Volume": volume["AAPL"]
}).loc[:observation_date].tail(60)

In [ ]:
# Validate the window
assert len(last_60) == 60
assert (last_60["Close"] > 0).all()
assert (last_60["Volume"] > 0).all()
assert last_60.index.equals(last_60_events.index)

In [ ]:
# Calculate approximate daily traded value
last_60["Daily_Traded_Value"] = (
    last_60["Close"] * last_60["Volume"])

In [ ]:
# Calculate Monetary
monetary = last_60["Daily_Traded_Value"].sum()

print("Window start:", last_60.index[0].date())
print("Window end:", last_60.index[-1].date())
print("Trading sessions:", len(last_60))
print(f"Monetary (USD): ${monetary:,.2f}")

Window start: 2024-10-07
Window end: 2024-12-31
Trading sessions: 60
Monetary (USD): $623,303,377,706.58


In [ ]:

display(last_60.tail(5))

,Close,Volume,Daily_Traded_Value
Date,,,
2024-12-24,258.200012,23234700,5.999200e+09
2024-12-26,259.019989,27237100,7.054953e+09
2024-12-27,255.589996,42355300,1.082559e+10
2024-12-30,252.199997,35557500,8.967601e+09
2024-12-31,250.419998,39480700,9.886757e+09


In [ ]:
# Identify high-volume days
clean_volume = volume.where(volume > 0)
previous_20 = clean_volume.shift(1).rolling(
    20, min_periods=20).mean()
high_volume_all = clean_volume > (2 * previous_20)

In [ ]:
# Find the most recent qualifying event
last_event = high_volume_all.apply(
    lambda s: s.index[s].max() if s.any() else pd.NaT)

In [ ]:
# Build the RFM features
rfm = pd.DataFrame({
    "Recency_Days": (observation_date - last_event).dt.days,
    "Frequency": high_volume_all.tail(60).sum(),
    "Monetary_USD": (close.tail(60) * volume.tail(60)).sum(
        min_count=60)
})

In [ ]:
# Retain shares with complete final 80 sessions
eligible = checks.index[checks["Last80_valid"]]
rfm = rfm.loc[eligible]

In [ ]:
# Review and save the results
print("Observation date:", observation_date.date())
print("Eligible shares:", len(rfm))
print("Shares with no high-volume event:",
      rfm["Recency_Days"].isna().sum())

display(rfm.head(10))
display(rfm.describe())

rfm.to_csv(folder / "rfm_features.csv")

Observation date: 2024-12-31
Eligible shares: 498
Shares with no high-volume event: 0


,Recency_Days,Frequency,Monetary_USD
MMM,11.0,4,2.599146e+10
AOS,11.0,2,4.748980e+09
ABT,11.0,1,3.523027e+10
ABBV,11.0,3,6.837054e+10
ACN,11.0,2,5.199430e+10
ADBE,18.0,3,1.030403e+11
AMD,18.0,4,3.131659e+11
AES,54.0,3,1.119578e+10
AFL,11.0,2,1.328456e+10
A,11.0,2,1.400180e+10


,Recency_Days,Frequency,Monetary_USD
count,498.000000,498.000000,4.980000e+02
mean,20.510040,3.004016,3.961352e+10
std,21.247722,1.327614,1.278859e+11
min,4.000000,0.000000,1.297483e+09
25%,11.000000,2.000000,1.025102e+10
50%,11.000000,3.000000,1.679667e+10
75%,18.000000,4.000000,3.287502e+10
max,222.000000,8.000000,1.847697e+12


In [ ]:

# Verify all-stock RFM against AAPL example

print(rfm.loc["AAPL"])

assert rfm.loc["AAPL", "Recency_Days"] == recency
assert rfm.loc["AAPL", "Frequency"] == frequency
assert np.isclose(
    rfm.loc["AAPL", "Monetary_USD"], monetary
)

print("AAPL RFM verification passed.")


Recency_Days    1.100000e+01
Frequency       2.000000e+00
Monetary_USD    6.233034e+11
Name: AAPL, dtype: float64
AAPL RFM verification passed.


1.5 Metric interpretation and supporting evidence

In [ ]:
# 1.5 Inclusion and exclusion log

exclusion_log = pd.DataFrame(index=tickers)
exclusion_log["Reason"] = "Included"

In [ ]:
# No downloaded history
exclusion_log.loc[
    checks["Valid_days"] == 0, "Reason"
] = "No downloaded data"

In [ ]:
# Insufficient recent history
exclusion_log.loc[
    (exclusion_log["Reason"] == "Included") &
    (~checks["Last80_valid"]), "Reason"
] = "Incomplete final 80 sessions"

In [ ]:
# Insufficient weekly observations for beta
exclusion_log.loc[
    (exclusion_log["Reason"] == "Included") &
    (features["Weekly_Beta"].isna()), "Reason"
] = "Insufficient beta history"

In [ ]:
# Missing RFM features
exclusion_log.loc[
    (exclusion_log["Reason"] == "Included") &
    (rfm["Recency_Days"].reindex(tickers).isna()), "Reason"
] = "No qualifying high-volume event"

In [ ]:
exclusion_log.loc[
    (exclusion_log["Reason"] == "Included") &
    (rfm["Monetary_USD"].reindex(tickers).isna()), "Reason"
] = "Incomplete Monetary data"

In [ ]:
# Common shares for clustering
common_tickers = exclusion_log.index[
    exclusion_log["Reason"] == "Included"
]

print("Inclusion and exclusion summary:")
print(exclusion_log["Reason"].value_counts())

print("\nExcluded shares:")
display(exclusion_log[exclusion_log["Reason"] != "Included"])

print("Common shares for clustering:", len(common_tickers))

exclusion_log.to_csv(folder / "inclusion_exclusion_log.csv")


Inclusion and exclusion summary:
Reason
Included                     495
No downloaded data             5
Insufficient beta history      3
Name: count, dtype: int64

Excluded shares:


,Reason
FDXF,No downloaded data
GEV,Insufficient beta history
HONA,No downloaded data
Q,No downloaded data
RDDT,Insufficient beta history
SNDK,No downloaded data
SOLV,Insufficient beta history
VYLR,No downloaded data


Common shares for clustering: 495
